In [ ]:
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel

from PIL import Image, ImageOps
import cv2
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm.auto import tqdm
import traceback
import numpy as np

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
HF_MODEL_NAME="facebook/dinov2-base"
MODEL_PATH = "../models/dino2b_finetuned.pth"
EMBEDDING_DIM  = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def get_image(image_path):
    image = Image.open(image_path)
    return image

def img2vec(image, processor, model):
    inputs = processor(image, return_tensors = "pt").to(device)
    pixel_values = inputs["pixel_values"].to(device)
    with torch.no_grad():
        embedding = model(pixel_values)
    return embedding.squeeze(0).cpu().numpy().tolist()

def pad_to_square(img, background_color=(255, 255, 255)):
    width, height = img.size
    if width == height:
        return img
    elif width > height:
        padding = (0, (width - height) // 2, 0, (width - height + 1) // 2)
    else:
        padding = ((height - width) // 2, 0, (height - width + 1) // 2, 0)
    return ImageOps.expand(img, padding, fill=background_color)

def gray_image(img):
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_img = cv2.merge([gray, gray, gray])
    gray_img = Image.fromarray(gray_img)
    return gray_img

def preprocess_image(image):
    img = image.convert("RGB")
    padded_img = pad_to_square(img, background_color=(255, 255, 255))
    gray_img = gray_image(padded_img)
    return gray_img

def path2vec(image_path, processor, model):
    try:
        img=get_image(image_path)
        img=preprocess_image(img)
        vec=img2vec(img, processor, model)
        return vec
    except Exception as e:
        print(f"Error processing image: {image_path}\nError: {e}")
        traceback.print_exc()
        return None

In [ ]:
class DinoV2HFWithProjection(nn.Module):
    def __init__(self, model_name_hf=HF_MODEL_NAME, embedding_dim=EMBEDDING_DIM):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name_hf)
        self.backbone_feature_dim = self.backbone.config.hidden_size
        self.projection_head = nn.Linear(self.backbone_feature_dim, embedding_dim)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        features = outputs.last_hidden_state[:, 0]
        embeddings = self.projection_head(features)
        return embeddings

In [ ]:
model = DinoV2HFWithProjection()
checkpoint = torch.load(MODEL_PATH)
model.load_state_dict(checkpoint)
model.eval().to(device)

In [ ]:
_ = model(torch.rand(1, 3, 224, 224).to(device))

# Vectorize

In [ ]:
import os
from tqdm.notebook import tqdm

tqdm.pandas()

image_paths = []
main_folder = "../data/Designs_bw"

for root, dirs, files in os.walk(main_folder):
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp")):
            image_paths.append(os.path.join(root, file))

print(image_paths)

In [ ]:
df = pd.DataFrame(image_paths, columns=["image_paths"])

In [ ]:
from tqdm.notebook import tqdm

tqdm.pandas()

df["vectors"] = df["image_paths"].progress_apply(lambda x: path2vec(x, processor, model))

In [ ]:
df.info()

In [ ]:
df.to_csv("train_vectors.csv", index=False)

# Test

In [ ]:
path = '../data/Designs_bw/_ (10)/Screenshot 2025-05-19 151905.png'
vector =  path2vec(path, processor, model)

In [ ]:
print(f"vector:{vector}, length:{len(vector)}, datatpe:{type(vector)}")